# Chapter 04 — Normalization, the Offset Map, Language & Shadow Text

*Where we are:* between parsing and chunking sits the step that makes search work **without
destroying provenance**.

```
document model →[ normalize + OffsetMap (shadow text) ]→ chunking → retrieval
```

The load-bearing idea: we search over a **normalized "shadow" text**, but every hit must resolve
back to the **exact original** span (and thence to an XML node or PDF box). The `OffsetMap` makes
normalization *reversible*.

In [1]:
# === Chapter 04 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 04 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 04 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 11. Unicode normalization — NFC / NFD / NFKC / NFKD

The same visible text can have different code-point sequences. **NFC** composes (é as one code
point); **NFD** decomposes (e + combining accent); **NFK*** additionally apply *compatibility*
mappings (½→1⁄2, ﬁ→fi) — useful for lexical matching but **destructive** (they change meaning/
length). For provenance we prefer **NFC**: it canonicalizes without throwing information away.

In [2]:
import unicodedata, pandas as pd
sample = "café \uFB01le ½ x\u00b2 Mert \u00d6z"   # composed é, ﬁ ligature, ½, superscript 2, Öz
rows = []
for form in ["NFC", "NFD", "NFKC", "NFKD"]:
    n = unicodedata.normalize(form, sample)
    rows.append({"form": form, "len": len(n), "text": n})
print("original len:", len(sample), "|", repr(sample))
pd.DataFrame(rows)

original len: 21 | 'café ﬁle ½ x² Mert Öz'


,form,len,text
0,NFC,21,café ﬁle ½ x² Mert Öz
1,NFD,23,café ﬁle ½ x² Mert Öz
2,NFKC,24,café file 1⁄2 x2 Mert Öz
3,NFKD,26,café file 1⁄2 x2 Mert Öz


Notice NFKC/NFKD *expand* ½→"1⁄2" and ﬁ→"fi" — great for a lexical index, but the character
count changed, so naive offsets into normalized text no longer point at the right original
characters. That is precisely the problem the **OffsetMap** solves.

## 12. The OffsetMap (this is critical)

Built per *normalization cluster* (a base char + its combining marks), the map records, for every
normalized character, the **original span** it came from. Reverse lookup of any normalized span
returns the minimal covering original span — even when normalization expanded or contracted
characters.

In [3]:
from patentrag.normalize import OffsetMap, normalize_text
om = OffsetMap.build("re\u0301sume\u0301 \uFB01le", "NFC")   # decomposed accents + ligature
print("original  :", repr(om.original))
print("normalized:", repr(om.normalized))
# recover the word 'résumé' from a normalized span
s = om.normalized.index("r"); e = om.normalized.index(" ")
print(f"normalized[{s}:{e}] = {om.normalized[s:e]!r}")
print("  -> original span:", om.to_original_span(s, e), "-> original chars:", repr(om.recover_original(s, e)))

original  : 'résumé ﬁle'
normalized: 'résumé ﬁle'
normalized[0:6] = 'résumé'
  -> original span: (0, 8) -> original chars: 'résumé'


In [4]:
# Exhaustive round-trip test over hard cases: every normalized span recovers a valid original span.
cases = [("NFC", "café résumé"), ("NFKC", "½ \uFB01le \u2075 x\u00b2"),
         ("NFC", "Mert \u00d6z and Herwig H\u00e4le"), ("NFKC", "\u2460 \u339C")]  # ①, ㎜
ok = True
for form, s in cases:
    m = OffsetMap.build(s, form)
    assert m.normalized == normalize_text(s, form)
    for a in range(len(m.normalized) + 1):
        for b in range(a, len(m.normalized) + 1):
            o0, o1 = m.to_original_span(a, b)
            if a != b:
                assert m.normalized[a:b] in unicodedata.normalize(form, s[o0:o1])
print("Exhaustive OffsetMap round-trip: PASS for", len(cases), "Unicode edge cases")
# ligature expands (1->2) and decomposed accent contracts (2->1):
print("ﬁ expands:", OffsetMap.build("\uFB01x","NFKC").to_original_span(0,2), "-> recovers", repr(OffsetMap.build("\uFB01x","NFKC").recover_original(0,1)))
print("é contracts:", OffsetMap.build("e\u0301","NFC").to_original_span(0,1))

Exhaustive OffsetMap round-trip: PASS for 4 Unicode edge cases
ﬁ expands: (0, 1) -> recovers 'ﬁ'
é contracts: (0, 2)


**Why this matters:** citability. A retrieved passage lives in shadow-text offsets; to cite it we
recover the original span and follow the anchor to the XML node / PDF bounding box. Without a
reversible map, normalization silently breaks every citation.

## 13. Boilerplate removal — structurally, without losing provenance

Patents carry boilerplate (priority claims, incorporation-by-reference). We **detect** it
heuristically and can *demote* it in retrieval, but we never delete the source — the section (and
its anchor) stays, so provenance is intact.

In [5]:
from patentrag.normalize import is_boilerplate
docs = bs.ensure("docs_canonical")
d = next(x for x in docs if any("PRIORITY" in s.heading.upper() or "CROSS" in s.heading.upper() for s in x.sections))
for s in d.sections[:6]:
    flag = "BOILERPLATE" if is_boilerplate(s.text) else "content"
    print(f"  [{flag:11}] {s.heading[:40]}")
print("\nWe TAG boilerplate (to de-weight in retrieval); the section + its anchor remain for provenance.")

  [content    ] ABSTRACT
  [BOILERPLATE] CROSS-REFERENCE TO RELATED APPLICATION
  [content    ] BACKGROUND
  [content    ] SUMMARY
  [content    ] BRIEF DESCRIPTION OF THE DRAWINGS
  [content    ] DETAILED DESCRIPTION

We TAG boilerplate (to de-weight in retrieval); the section + its anchor remain for provenance.


## 14. DOM anchoring — XPath round-trip

For XML sources, provenance is an **XPath** to the exact node. We retrieve a node, record its
absolute XPath, then navigate back to prove the anchor resolves.

In [6]:
from patentrag import parsing as P
from lxml import etree
tree = P.load_xml(bs.DATA / "xml" / "ST96_PatentPublication_Example.xml")
title_node = tree.xpath("//*[local-name()='InventionTitle']")[0]
xp = tree.getpath(title_node)             # absolute, namespace-prefixed XPath to this node
print("node XPath:", xp)
# navigate back — the prefixes in getpath()'s output must be supplied to resolve them
nsmap = {k: v for k, v in title_node.nsmap.items() if k}
back = tree.xpath(xp, namespaces=nsmap)[0]
print("round-trip resolves to same node:", back is title_node, "| text:", back.text[:60])

node XPath: /pat:PatentPublication/pat:BibliographicData/pat:InventionTitleBag/pat:InventionTitle
round-trip resolves to same node: True | text: Manufacturing method for room-temperature substrate bonding


## 15. Language identification

Patents are multilingual and language can vary **per chunk** (e.g. an English patent quoting a
foreign title). `langdetect` (deterministic with a fixed seed) gives ISO-639-1 codes.

In [7]:
from patentrag.normalize import detect_language
samples = {
    "corpus abstract (en)": d.abstract[:200],
    "French": "Un procédé de recherche de brevets utilisant des vecteurs denses.",
    "German": "Ein Verfahren zur Ähnlichkeitssuche in großen Vektordatenbanken.",
    "Japanese": "ベクトル検索を用いた特許検索のための方法。",
}
pd.DataFrame([{"text_sample": k, "detected": detect_language(v)} for k, v in samples.items()])

,text_sample,detected
0,corpus abstract (en),en
1,French,fr
2,German,de
3,Japanese,ja


## 16. Shadow text — original immutable, shadow searchable

The architecture: keep the **source text immutable**; derive a **shadow** (normalized, search-
optimized) representation with an OffsetMap back to the source. `normalize_document` produces the
`docs_normalized` artifact every later stage consumes.

In [8]:
nd = bs.ensure("docs_normalized")
one = next(n for n in nd if n.doc_id == d.doc_id)
sec = one.sections[0]
print("doc:", one.doc_id, "| section:", sec.heading)
print("original (head):", repr(sec.original[:70]))
print("shadow   (head):", repr(sec.shadow[:70]))
# prove reversibility on this real section: recover the first 40 shadow chars
print("recovered original for shadow[0:40]:", repr(sec.offset_map.recover_original(0, 40)))
print(f"\ndocs_normalized: {len(nd)} documents, each carrying per-section OffsetMaps")

doc: US10083169B1 | section: ABSTRACT
original (head): 'Methods, systems, and apparatus, including computer programs encoded o'
shadow   (head): 'Methods, systems, and apparatus, including computer programs encoded o'
recovered original for shadow[0:40]: 'Methods, systems, and apparatus, includi'

docs_normalized: 15 documents, each carrying per-section OffsetMaps


## Chapter invariants

In [9]:
# Reversibility holds on real corpus text, end to end.
for one in nd[:5]:
    for sec in one.sections:
        L = min(50, len(sec.shadow))
        if L:
            o0, o1 = sec.offset_map.to_original_span(0, L)
            assert sec.offset_map.recover_original(0, L) == sec.original[o0:o1]
assert detect_language(samples["French"]) == "fr"
assert normalize_text("e\u0301", "NFC") == "\u00e9"
print("All Chapter 04 invariants hold. docs_normalized artifact ready.")

All Chapter 04 invariants hold. docs_normalized artifact ready.


In [10]:
# === Chapter 04 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['langdetect', 'lxml', 'pandas']
print("Chapter 04 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 04 VALIDATION: PASS")

Chapter 04 — environment
  Python : 3.12.10 on Windows 11
  langdetect              : 1.0.9
  lxml                    : 6.1.1
  pandas                  : 3.0.2

CHAPTER 04 VALIDATION: PASS
